<a href="https://colab.research.google.com/github/liangchow/sade-geo/blob/main/Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Set Up Worksheet and Import Libraries

In [9]:
# Clone Gitub repository to Colab
from google.colab import drive
drive.mount('/content/drive')

!apt-get install git
!git clone https://github.com/liangchow/sade-geo.git

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
fatal: destination path 'sade-geo' already exists and is not an empty directory.


In [10]:
import requests
import json
from geopy.geocoders import Nominatim
from pyproj import Transformer

In [11]:
# Hazard URLs
LIQUEFACTION_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/CGS_Liquefaction_Zones/"
    "FeatureServer/0/query"
)

AP_FAULT_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/"
    "CGS_Alquist_Priolo_Fault_Zones/"
    "FeatureServer/0/query"
)

LANDSLIDE_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/"
    "CGS_Landslide_Zones/"
    "FeatureServer/0/query"
)

UNEVALUATED_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/"
    "CGS_SHZ_Unevaluated_Areas/"
    "FeatureServer/0/query"
)

GEOLOGY_URL = (
    "https://gis.conservation.ca.gov/server/rest/services/"
    "CGS/Geologic_Map_of_California/"
    "MapServer/12/query"
)

VS30_URL = (
    "https://gis.conservation.ca.gov/server/rest/services/"
    "CGS/MS48_Vs30_ShearWaveVelocity2022/"
    "ImageServer"
)

GIS_LAYERS = {
    "Liq": {"url": LIQUEFACTION_URL, "type": "boolean"},
    "AP Fault": {"url": AP_FAULT_URL, "type": "boolean"},
    "Landslide": {"url": LANDSLIDE_URL, "type": "boolean"},
    "Unevaluated": {"url": UNEVALUATED_URL, "type": "boolean"},
    "Geology": {"url": GEOLOGY_URL, "type": "attributes", "query_function": "geology",
        "fields": [
            "PTYPE",
            "GENERAL_LITHOLOGY",
            "AGE",
            "DESCRIPTION"
        ]},
    "Vs30": {"url": VS30_URL, "type": "raster"},
}

In [12]:
# FeatureServer query function
def query_feature_service(url, lat, lon, inSR=4326):

    params = {
        "f": "json",
        "geometry": f"{lon},{lat}",
        "geometryType": "esriGeometryPoint",
        "spatialRel": "esriSpatialRelIntersects",
        "inSR": inSR,
        "returnGeometry": "false",
        "outFields": "*"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()

    return r.json()


def get_layer_result(layer, lat, lon):
    # Raster layers (ImageServer)
    if layer["type"] == "raster":
        return query_vs30(lat, lon)

    # Special geometry handling for geology
    if layer.get("query_function") == "geology":
        result = query_geology(lat, lon)
    # Normal FeatureServer layers
    else:
        result = query_feature_service(
            layer["url"], lat, lon
        )
    features = result.get("features", [])

    if layer["type"] == "boolean":
        return len(features) > 0

    elif layer["type"] == "attributes":
        if not features:
            return None
        attrs = features[0]["attributes"]
        return {
            field: attrs.get(field)
            for field in layer["fields"]
        }

    else:
        return None

def get_all_services(lat, lon):

  results = {}

  for name, layer in GIS_LAYERS.items():
      results[name] = get_layer_result(
          layer,
          lat,
          lon
      )

  return results

In [13]:
# Geology MapServer query function
to_webmercator = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True
)

def query_geology(lat, lon):

    x, y = to_webmercator.transform(lon, lat)

    params = {
        "f": "json",
        "geometry": f"{x},{y}",
        "geometryType": "esriGeometryPoint",
        "spatialRel": "esriSpatialRelIntersects",
        "inSR": 102100,
        "returnGeometry": "false",
        "outFields": (
            "OBJECTID,"
            "PTYPE,"
            "GENERAL_LITHOLOGY,"
            "AGE,"
            "DESCRIPTION"
        )
    }

    r = requests.get(GEOLOGY_URL, params=params)
    r.raise_for_status()

    return r.json()

In [14]:
# Vs30 ImageServer query function
def query_vs30(lat, lon):

    params = {
        "f": "json",
        "geometry": f"{lon},{lat}",
        "geometryType": "esriGeometryPoint",
        "returnGeometry": "false"
    }

    r = requests.get(
        VS30_URL + "/identify",
        params=params
    )

    r.raise_for_status()

    data = r.json()

    return data.get("value")

In [21]:
# Geolocation
def get_location(lat, lon):
    # Initialize the geolocator
    geolocator = Nominatim(user_agent="city_lookup")
    location = geolocator.reverse((lat, lon), exactly_one=True)

    if location is None:
        return None

    address = location.raw.get("address", {})

    return {
        "address": location.address,
        "city": (
            address.get("city")
            or address.get("town")
            or address.get("village")
            or address.get("municipality")
        ),
        "state": address.get("state"),
        "country": address.get("country"),
        "postcode": address.get("postcode"),
    }

##Test Cell

In [27]:
# Example lat/long
lat = 37.352147
lon = -121.883908

get_location(lat, lon)

{'address': 'North 15th Street, Luna Park, San Jose, Santa Clara County, California, 95112, United States',
 'city': 'San Jose',
 'state': 'California',
 'country': 'United States',
 'postcode': '95112'}

In [28]:
get_all_services(lat, lon)

{'Liq': True,
 'AP Fault': False,
 'Landslide': False,
 'Unevaluated': False,
 'Geology': {'PTYPE': 'Q',
  'GENERAL_LITHOLOGY': 'marine and nonmarine (continental) sedimentary rocks',
  'AGE': 'Pleistocene-Holocene',
  'DESCRIPTION': 'Alluvium, lake, playa, and terrace deposits; unconsolidated and semi-consolidated. Mostly nonmarine, but includes marine deposits near the coast.'},
 'Vs30': '228.2'}

In [29]:
results = get_all_services(lat, lon)
location_info = get_location(lat, lon)

markdown_output = "### Geological Hazard Assessment\n\n"
markdown_output += f"Based on the provided latitude (`{lat}`) and longitude (`{lon}`):\n"
if location_info:
    markdown_output += f"Location: `{location_info.get('address', 'N/A')}`\n"
    markdown_output += f"City: `{location_info.get('city', 'N/A')}`\n"
    markdown_output += f"State: `{location_info.get('state', 'N/A')}`\n"
    markdown_output += f"Country: `{location_info.get('country', 'N/A')}`\n\n"
else:
    markdown_output += "No detailed location information available.\n\n"

# Boolean results
for key in ['Liq', 'AP Fault', 'Landslide', 'Unevaluated']:
    value = results.get(key, 'N/A')
    description = ''
    if key == 'Liq':
        description = '(Presence of liquefaction zones)' if value else '(No liquefaction zones detected)'
    elif key == 'AP Fault':
        description = '(Alquist-Priolo Fault zones detected)' if value else '(No Alquist-Priolo Fault zones detected)'
    elif key == 'Landslide':
        description = '(Landslide zones detected)' if value else '(No landslide zones detected)'
    elif key == 'Unevaluated':
        description = '(Area is unevaluated)' if value else '(Area has been evaluated)'
    markdown_output += f"- **{key}**: `{value}` {description}\n"

# Geology results
geology_data = results.get('Geology', {})
if geology_data:
    markdown_output += "- **Geology**:\n"
    for geo_key, geo_value in geology_data.items():
        markdown_output += f"  - **{geo_key}**: `{geo_value}`\n"

# Vs30 results
vs30_value = results.get('Vs30', 'N/A')
markdown_output += f"- **Vs30 (Shear-wave velocity)**: `{vs30_value}` (meters/second)\n"

print(markdown_output)

### Geological Hazard Assessment

Based on the provided latitude (`37.352147`) and longitude (`-121.883908`):
Location: `North 15th Street, Luna Park, San Jose, Santa Clara County, California, 95112, United States`
City: `San Jose`
State: `California`
Country: `United States`

- **Liq**: `True` (Presence of liquefaction zones)
- **AP Fault**: `False` (No Alquist-Priolo Fault zones detected)
- **Landslide**: `False` (No landslide zones detected)
- **Unevaluated**: `False` (Area has been evaluated)
- **Geology**:
  - **PTYPE**: `Q`
  - **GENERAL_LITHOLOGY**: `marine and nonmarine (continental) sedimentary rocks`
  - **AGE**: `Pleistocene-Holocene`
  - **DESCRIPTION**: `Alluvium, lake, playa, and terrace deposits; unconsolidated and semi-consolidated. Mostly nonmarine, but includes marine deposits near the coast.`
- **Vs30 (Shear-wave velocity)**: `228.2` (meters/second)

